# Đáp Án Mẫu — `exercises.md`

Notebook này dành cho **facilitator**, không phải cho người tham gia — nó chứa câu SQL mẫu và kết quả mong đợi cho từng câu hỏi trong `exercises.md` (Bước 3), chạy trực tiếp trên `data/workshop.duckdb` thật.

Dùng notebook này để:
- Kiểm tra nhanh xem một câu trả lời do AI sinh ra có "đúng" hay không (so kết quả).
- Có sẵn một câu SQL tham khảo nếu bạn cần giải thích cách ra kết quả cho người tham gia.

Với các câu **Nâng cao**, không có một câu SQL "đúng" duy nhất — SQL bên dưới chỉ là **một cách hợp lý** để trả lời, không phải đáp án chuẩn tuyệt đối. Skill của người tham gia có thể ra một câu SQL khác miễn là logic và con số hợp lý.


In [1]:
import duckdb
import pandas as pd

pd.set_option("display.max_rows", 30)
pd.set_option("display.width", 120)

con = duckdb.connect("../../data/workshop.duckdb", read_only=True)
con.sql("SELECT count(*) AS so_dong FROM transactions").df()


,so_dong
0,2000000


## Cấp độ Cơ bản — một bảng, tổng hợp đơn giản

### B1. Chúng ta có bao nhiêu khách hàng, phân theo giới tính?

In [2]:
con.sql('''
SELECT gender, COUNT(*) AS so_luong
FROM customers
GROUP BY gender
ORDER BY so_luong DESC
''').df()


,gender,so_luong
0,Male,31344
1,Female,27412
2,Other,1244


### B2. Số dư tài khoản trung bình của mỗi loại tài khoản (`account_type`) là bao nhiêu?

In [3]:
con.sql('''
SELECT account_type, ROUND(AVG(balance), 2) AS so_du_trung_binh
FROM accounts
GROUP BY account_type
ORDER BY so_du_trung_binh DESC
''').df()


,account_type,so_du_trung_binh
0,NRI,46657.61
1,Savings,45997.15
2,Current,45744.84
3,Salary,45193.94
4,Fixed Deposit,45019.02


### B3. Có bao nhiêu ticket hỗ trợ đang ở trạng thái \"Open,\" phân theo loại vấn đề?

In [4]:
con.sql('''
SELECT issue_type, COUNT(*) AS so_luong
FROM support_tickets
WHERE status = 'Open'
GROUP BY issue_type
ORDER BY so_luong DESC
''').df()


,issue_type,so_luong
0,Card Blocked,318
1,Account Statement,311
2,Loan Query,310
3,Fraud Report,309
4,Wrong Debit,300
5,Interest Query,298
6,KYC Update,293
7,App Login Issue,291
8,Cheque Bounce,289
9,Net Banking Issue,268


### B4. Liệt kê 10 khoản vay có số tiền gốc (`loan_amount`) lớn nhất, kèm loại vay và trạng thái.

In [5]:
con.sql('''
SELECT loan_id, loan_type, loan_amount, status
FROM loans
ORDER BY loan_amount DESC
LIMIT 10
''').df()


,loan_id,loan_type,loan_amount,status
0,19771,Business Loan,4016132.25,Active
1,13012,Education Loan,3565151.75,Active
2,2236,Auto Loan,3230783.99,Active
3,9056,Home Loan,3211860.83,Active
4,1453,Gold Loan,3211067.07,Active
5,17558,Business Loan,3192082.57,Active
6,11530,Business Loan,3168213.61,Closed
7,3059,Business Loan,3033847.06,Active
8,14984,Business Loan,3019203.58,Defaulted
9,8641,Auto Loan,2942986.32,Active


### B5. Trong số các giao dịch thẻ (`card_transactions`), bao nhiêu phần trăm bị đánh dấu gian lận (`is_fraud`)?

In [6]:
con.sql('''
SELECT
    COUNT(*) AS tong_giao_dich,
    SUM(is_fraud) AS so_giao_dich_gian_lan,
    ROUND(100.0 * SUM(is_fraud) / COUNT(*), 3) AS ty_le_gian_lan_pct
FROM card_transactions
''').df()


,tong_giao_dich,so_giao_dich_gian_lan,ty_le_gian_lan_pct
0,3000000,14954.0,0.498


## Cấp độ Trung cấp — join 2 bảng, GROUP BY có điều kiện

### I1. Chi nhánh nào đã xử lý tổng số tiền giao dịch (`transactions`) cao nhất?

In [7]:
con.sql('''
SELECT b.branch_id, b.branch_name, ROUND(SUM(t.amount), 2) AS tong_so_tien
FROM transactions t
JOIN accounts a ON t.account_id = a.account_id
JOIN branches b ON a.branch_id = b.branch_id
GROUP BY b.branch_id, b.branch_name
ORDER BY tong_so_tien DESC
LIMIT 10
''').df()


,branch_id,branch_name,tong_so_tien
0,58,Delhi Branch 4,88451045.36
1,132,Jaipur Branch 6,88363724.65
2,15,Chandigarh Branch 6,87672615.90
3,148,Bengaluru Branch 4,86942085.68
4,54,Ahmedabad Branch 9,86872570.83
5,66,Chennai Branch 3,86744668.98
6,12,Indore Branch 3,85888521.06
7,150,Mumbai Branch 6,85826523.72
8,29,Jaipur Branch 2,85745135.61
9,77,Chennai Branch 5,85739107.63


### I2. Với các khách hàng gia nhập năm 2023, điểm tín dụng trung bình của họ so với khách hàng gia nhập năm 2015 như thế nào?

In [8]:
con.sql('''
SELECT
    EXTRACT(YEAR FROM join_date) AS nam_gia_nhap,
    COUNT(*) AS so_khach_hang,
    ROUND(AVG(credit_score), 2) AS diem_tin_dung_trung_binh
FROM customers
WHERE EXTRACT(YEAR FROM join_date) IN (2023, 2015)
GROUP BY nam_gia_nhap
ORDER BY nam_gia_nhap
''').df()


,nam_gia_nhap,so_khach_hang,diem_tin_dung_trung_binh
0,2015,2854,602.19
1,2023,2753,604.53


### I3. Tỷ lệ thanh toán trễ (`late_payment_flag`) trên tổng số lượt thanh toán là bao nhiêu, phân theo từng loại khoản vay (`loan_type`)?

In [9]:
con.sql('''
SELECT
    l.loan_type,
    COUNT(*) AS so_lan_thanh_toan,
    SUM(lp.late_payment_flag) AS so_lan_tre,
    ROUND(100.0 * SUM(lp.late_payment_flag) / COUNT(*), 2) AS ty_le_tre_pct
FROM loan_payments lp
JOIN loans l ON lp.loan_id = l.loan_id
GROUP BY l.loan_type
ORDER BY ty_le_tre_pct DESC
''').df()


,loan_type,so_lan_thanh_toan,so_lan_tre,ty_le_tre_pct
0,Auto Loan,99013,12067.0,12.19
1,Home Loan,100056,12096.0,12.09
2,Education Loan,101448,12242.0,12.07
3,Personal Loan,98627,11812.0,11.98
4,Gold Loan,100571,11939.0,11.87
5,Business Loan,100285,11882.0,11.85


### I4. 10 khách hàng nào mở nhiều ticket hỗ trợ nhất, và loại vấn đề (`issue_type`) phổ biến nhất của mỗi người là gì?

In [10]:
con.sql('''
WITH counts AS (
    SELECT customer_id, issue_type, COUNT(*) AS cnt
    FROM support_tickets
    GROUP BY customer_id, issue_type
),
ranked AS (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY cnt DESC) AS rn
    FROM counts
),
totals AS (
    SELECT customer_id, COUNT(*) AS tong_so_ticket
    FROM support_tickets
    GROUP BY customer_id
)
SELECT
    t.customer_id,
    c.name,
    t.tong_so_ticket,
    r.issue_type AS van_de_pho_bien_nhat
FROM totals t
JOIN ranked r ON t.customer_id = r.customer_id AND r.rn = 1
JOIN customers c ON c.customer_id = t.customer_id
ORDER BY t.tong_so_ticket DESC
LIMIT 10
''').df()


,customer_id,name,tong_so_ticket,van_de_pho_bien_nhat
0,12450,Ajay Malhotra,5,Account Statement
1,55858,Sanjay Pillai,5,Card Blocked
2,36989,Robert Smith,5,Card Blocked
3,38478,Kiran Davis,5,Account Statement
4,48400,Karen Reddy,5,Wrong Debit
5,10609,Kiran Joshi,5,Fraud Report
6,6799,Richard Singh,4,Cheque Bounce
7,11346,Rahul Garcia,4,Account Statement
8,8900,Ajay Martinez,4,Account Statement
9,10464,Priya Mehta,4,KYC Update


### I5. Bao nhiêu phần trăm giao dịch thẻ bị đánh dấu gian lận, phân theo loại thẻ (`card_type`)?

In [11]:
con.sql('''
SELECT
    c.card_type,
    COUNT(*) AS so_giao_dich,
    ROUND(100.0 * SUM(ct.is_fraud) / COUNT(*), 3) AS ty_le_gian_lan_pct
FROM card_transactions ct
JOIN cards c ON ct.card_id = c.card_id
GROUP BY c.card_type
ORDER BY ty_le_gian_lan_pct DESC
''').df()


,card_type,so_giao_dich,ty_le_gian_lan_pct
0,Credit - Classic,752873,0.507
1,Credit - Gold,386252,0.499
2,Debit,1647839,0.497
3,Credit - Platinum,213036,0.477


## Cấp độ Nâng cao — nhiều bảng, CTE/window function, suy luận nghiệp vụ

Nhắc lại: các câu này không có một đáp án SQL duy nhất. SQL dưới đây là ví dụ tham khảo, có nêu rõ giả định.

### A1. Xếp hạng rủi ro khách hàng

Giả định về cách tính điểm rủi ro (0–100, càng cao càng rủi ro), chỉ là một cách hợp lý trong nhiều cách:
- 40% trọng số: điểm tín dụng càng thấp → rủi ro càng cao (`(100 - credit_score/900*100) * 0.4`)
- 40% trọng số: tỷ lệ thanh toán trễ càng cao → rủi ro càng cao (`ty_le_tre_pct * 0.4`)
- 20% trọng số còn lại: mỗi ticket "Fraud Report" cộng thêm điểm rủi ro, giới hạn ở 5 ticket (`LEAST(so_ticket_fraud, 5) * 4`)

In [12]:
con.sql('''
WITH loan_scope AS (
    SELECT loan_id, customer_id
    FROM loans
    WHERE status IN ('Active', 'Defaulted')
),
payments_agg AS (
    SELECT
        ls.customer_id,
        SUM(lp.amount_paid) AS tong_da_thanh_toan,
        COUNT(*) AS so_lan_thanh_toan,
        SUM(lp.late_payment_flag) AS so_lan_tre
    FROM loan_scope ls
    JOIN loan_payments lp ON lp.loan_id = ls.loan_id
    GROUP BY ls.customer_id
),
fraud_tickets AS (
    SELECT customer_id, COUNT(*) AS so_ticket_fraud
    FROM support_tickets
    WHERE issue_type = 'Fraud Report'
    GROUP BY customer_id
),
combined AS (
    SELECT
        c.customer_id,
        c.name,
        c.credit_score,
        COALESCE(p.tong_da_thanh_toan, 0) AS tong_da_thanh_toan,
        ROUND(COALESCE(100.0 * p.so_lan_tre / NULLIF(p.so_lan_thanh_toan, 0), 0), 2) AS ty_le_tre_pct,
        COALESCE(f.so_ticket_fraud, 0) AS so_ticket_fraud
    FROM customers c
    JOIN payments_agg p ON p.customer_id = c.customer_id
    LEFT JOIN fraud_tickets f ON f.customer_id = c.customer_id
)
SELECT
    *,
    ROUND(
        (100 - credit_score / 900.0 * 100) * 0.4
        + ty_le_tre_pct * 0.4
        + LEAST(so_ticket_fraud, 5) * 4
    , 2) AS diem_rui_ro
FROM combined
ORDER BY diem_rui_ro DESC
LIMIT 20
''').df()


,customer_id,name,credit_score,tong_da_thanh_toan,ty_le_tre_pct,so_ticket_fraud,diem_rui_ro
0,267,Kiran Williams,343,230778.50,41.38,0,41.31
1,57023,Elizabeth Smith,336,318052.06,40.00,0,41.07
2,28427,Sarah Gupta,327,389872.55,36.36,0,40.01
3,34799,Anita Gupta,316,157965.12,25.00,1,39.96
4,9592,Anjali Jones,302,218426.23,33.33,0,39.91
5,20089,Pooja Martinez,385,259385.09,42.11,0,39.73
6,59144,Joseph Mehta,360,170477.77,37.50,0,39.00
7,56132,Robert Iyer,300,198005.42,30.77,0,38.97
8,22030,Rahul Sharma,301,256709.22,30.43,0,38.79
9,41431,Priya Garcia,303,361636.15,30.43,0,38.71


### A2. Tăng trưởng theo tháng

"24 tháng gần nhất" được tính theo `MAX(txn_date)` thực tế trong bảng `transactions`, không phải ngày hôm nay.

In [13]:
con.sql('''
WITH monthly AS (
    SELECT date_trunc('month', txn_date) AS thang, SUM(amount) AS tong_amount
    FROM transactions
    GROUP BY thang
),
recent AS (
    SELECT thang, tong_amount
    FROM monthly
    WHERE thang >= date_trunc('month', (SELECT MAX(txn_date) FROM transactions)) - INTERVAL '23 months'
)
SELECT
    thang,
    tong_amount,
    LAG(tong_amount) OVER (ORDER BY thang) AS thang_truoc_do,
    ROUND(
        100.0 * (tong_amount - LAG(tong_amount) OVER (ORDER BY thang))
        / NULLIF(LAG(tong_amount) OVER (ORDER BY thang), 0)
    , 2) AS phan_tram_tang_truong
FROM recent
ORDER BY thang
''').df()


,thang,tong_amount,thang_truoc_do,phan_tram_tang_truong
0,2024-07-01,1.213944e+08,NaN,NaN
1,2024-08-01,1.208802e+08,1.213944e+08,-0.42
2,2024-09-01,1.167048e+08,1.208802e+08,-3.45
3,2024-10-01,1.197707e+08,1.167048e+08,2.63
4,2024-11-01,1.164721e+08,1.197707e+08,-2.75
5,2024-12-01,1.200050e+08,1.164721e+08,3.03
6,2025-01-01,1.210197e+08,1.200050e+08,0.85
7,2025-02-01,1.114009e+08,1.210197e+08,-7.95
8,2025-03-01,1.206830e+08,1.114009e+08,8.33
9,2025-04-01,1.164571e+08,1.206830e+08,-3.50


### A3. Top 5% khách hàng theo tổng giá trị giao dịch

Cộng `transactions` (cấp tài khoản) và `card_transactions` (cấp thẻ) theo từng khách hàng riêng biệt — không `UNION` trực tiếp hai bảng vì khác grain. Chỉ tính khách hàng có ít nhất một tài khoản "Active". Dùng `PERCENT_RANK()` để lấy đúng phân vị 5% trên cùng.

In [14]:
con.sql('''
WITH acct_totals AS (
    SELECT a.customer_id, SUM(t.amount) AS acct_amount
    FROM transactions t
    JOIN accounts a ON t.account_id = a.account_id
    GROUP BY a.customer_id
),
card_totals AS (
    SELECT c.customer_id, SUM(ct.amount) AS card_amount
    FROM card_transactions ct
    JOIN cards c ON ct.card_id = c.card_id
    GROUP BY c.customer_id
),
active_customers AS (
    SELECT DISTINCT customer_id FROM accounts WHERE status = 'Active'
),
combined AS (
    SELECT
        ac.customer_id,
        COALESCE(a.acct_amount, 0) + COALESCE(ct.card_amount, 0) AS tong_gia_tri
    FROM active_customers ac
    LEFT JOIN acct_totals a ON a.customer_id = ac.customer_id
    LEFT JOIN card_totals ct ON ct.customer_id = ac.customer_id
),
ranked AS (
    SELECT *, PERCENT_RANK() OVER (ORDER BY tong_gia_tri) AS pct_rank
    FROM combined
),
top5 AS (
    SELECT customer_id, tong_gia_tri FROM ranked WHERE pct_rank >= 0.95
)
SELECT
    b.branch_id,
    b.branch_name,
    COUNT(DISTINCT t.customer_id) AS so_khach_hang_top5
FROM top5 t
JOIN accounts a ON a.customer_id = t.customer_id
JOIN branches b ON a.branch_id = b.branch_id
GROUP BY b.branch_id, b.branch_name
ORDER BY so_khach_hang_top5 DESC
LIMIT 10
''').df()


,branch_id,branch_name,so_khach_hang_top5
0,103,Kochi Branch 4,82
1,47,Kolkata Branch 2,80
2,34,Jaipur Branch 7,77
3,58,Delhi Branch 4,77
4,62,Bhopal Branch 8,76
5,52,Delhi Branch 7,76
6,150,Mumbai Branch 6,76
7,146,Kochi Branch 2,76
8,136,Chennai Branch 1,75
9,54,Ahmedabad Branch 9,75


### A4. Nợ xấu theo chi nhánh so với nhân sự

"Nợ xấu" = khoản vay ở trạng thái "Defaulted" hoặc "Written Off". So sánh tỷ lệ này với số nhân viên vai trò "Loan Officer" đang làm việc tại từng chi nhánh.

In [15]:
con.sql('''
WITH loan_stats AS (
    SELECT
        branch_id,
        COUNT(*) AS tong_khoan_vay,
        SUM(CASE WHEN status IN ('Defaulted', 'Written Off') THEN 1 ELSE 0 END) AS so_no_xau
    FROM loans
    GROUP BY branch_id
),
officer_counts AS (
    SELECT branch_id, COUNT(*) AS so_loan_officer
    FROM employees
    WHERE role = 'Loan Officer'
    GROUP BY branch_id
)
SELECT
    b.branch_id,
    b.branch_name,
    ls.tong_khoan_vay,
    ls.so_no_xau,
    ROUND(100.0 * ls.so_no_xau / ls.tong_khoan_vay, 2) AS ty_le_no_xau_pct,
    COALESCE(oc.so_loan_officer, 0) AS so_loan_officer
FROM branches b
JOIN loan_stats ls ON ls.branch_id = b.branch_id
LEFT JOIN officer_counts oc ON oc.branch_id = b.branch_id
ORDER BY ty_le_no_xau_pct DESC
LIMIT 15
''').df()


,branch_id,branch_name,tong_khoan_vay,so_no_xau,ty_le_no_xau_pct,so_loan_officer
0,104,Mumbai Branch 5,111,19.0,17.12,1
1,89,Bhopal Branch 8,152,24.0,15.79,5
2,82,Kolkata Branch 1,147,23.0,15.65,1
3,95,Nagpur Branch 5,167,26.0,15.57,1
4,75,Chandigarh Branch 3,134,20.0,14.93,0
5,61,Nagpur Branch 7,141,21.0,14.89,0
6,141,Indore Branch 6,138,20.0,14.49,3
7,5,Kolkata Branch 5,139,20.0,14.39,4
8,146,Kochi Branch 2,141,20.0,14.18,1
9,84,Kochi Branch 3,142,20.0,14.08,1


## Mở rộng — Tech Salary (Bước 7, tùy chọn)

Câu hỏi mẫu ở đây tương ứng với `exercises.md` (Bước 7). Dùng một kết nối riêng vì đây là một file `.duckdb` độc lập, không liên quan tới `data/workshop.duckdb` ở trên.

In [16]:
con2 = duckdb.connect("../../data/tech_salary.duckdb", read_only=True)
con2.sql("SELECT table_name, estimated_size AS so_dong FROM duckdb_tables()").df()


,table_name,so_dong
0,global_tech_market_2026,12003
1,usajobs_tech_roles_2026,2997


### T1. Mức lương trung bình cho vị trí Data Scientist

In [17]:
con2.sql('''
WITH all_jobs AS (
    SELECT * FROM global_tech_market_2026
    UNION ALL
    SELECT * FROM usajobs_tech_roles_2026
)
SELECT
    job_title,
    ROUND(AVG((salary_min_usd + salary_max_usd) / 2.0), 0) AS luong_trung_binh_usd,
    COUNT(*) AS so_tin_dang
FROM all_jobs
WHERE job_title = 'Data Scientist'
GROUP BY job_title
''').df()


,job_title,luong_trung_binh_usd,so_tin_dang
0,Data Scientist,122723.0,769


### T2. 3 tổ hợp công nghệ (`tech_stack`) xuất hiện trong nhiều tin đăng nhất

In [18]:
con2.sql('''
WITH all_jobs AS (
    SELECT * FROM global_tech_market_2026
    UNION ALL
    SELECT * FROM usajobs_tech_roles_2026
)
SELECT tech_stack, COUNT(*) AS so_tin_dang
FROM all_jobs
GROUP BY tech_stack
ORDER BY so_tin_dang DESC
LIMIT 3
''').df()


,tech_stack,so_tin_dang
0,"Java, Spring Boot, Kafka, PostgreSQL",1555
1,"Rust, WebAssembly, System Architecture",1548
2,"Ruby on Rails, Redis, Heroku",1528


### T3. So sánh mức lương tối đa trung bình: USAJOBS vs. toàn bộ thị trường tech

In [19]:
con2.sql('''
SELECT 'USAJOBS' AS nguon, ROUND(AVG(salary_max_usd), 0) AS luong_toi_da_trung_binh_usd
FROM usajobs_tech_roles_2026
UNION ALL
SELECT 'Toan bo thi truong (global_tech_market_2026)' AS nguon, ROUND(AVG(salary_max_usd), 0) AS luong_toi_da_trung_binh_usd
FROM global_tech_market_2026
''').df()


,nguon,luong_toi_da_trung_binh_usd
0,USAJOBS,143374.0
1,Toan bo thi truong (global_tech_market_2026),141565.0
